In [1]:
import sys
from pathlib import Path
import os

# Add the project root to Python path so imports work from subdirectories
# Try multiple methods to find the project root
current_dir = Path(os.getcwd())

# Method 1: Go up from current directory (assumes we're in VarianceDecisionTree/process_data)
project_root = current_dir.parent.parent

# Method 2: If that doesn't have main.py, try to find it by looking for main.py
if not (project_root / "main.py").exists():
    # Walk up until we find main.py
    test_path = current_dir
    while test_path != test_path.parent:
        if (test_path / "main.py").exists():
            project_root = test_path
            break
        test_path = test_path.parent

project_root_str = str(project_root.resolve())

# Remove project_root from path if it's there, then add it at the beginning
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)

# Import utils from the project root (not the installed package)
# Force reload if already imported to avoid conflicts
if 'utils' in sys.modules:
    # Check if it's the wrong utils (from site-packages)
    utils_module = sys.modules['utils']
    utils_file = getattr(utils_module, '__file__', None)
    if utils_file and 'site-packages' in str(utils_file):
        del sys.modules['utils']
        # Also remove any submodules
        keys_to_remove = [k for k in sys.modules.keys() if k.startswith('utils.')]
        for k in keys_to_remove:
            del sys.modules[k]

import utils
# Verify we imported the correct utils module (should have 'announce' and 'unzip' functions)
if not hasattr(utils, 'announce'):
    utils_file = getattr(utils, '__file__', 'unknown location')
    raise ImportError(f"Wrong utils module imported! Expected local utils.py with 'announce' function. "
                     f"Got: {utils_file}")
if not hasattr(utils, 'unzip'):
    utils_file = getattr(utils, '__file__', 'unknown location')
    raise ImportError(f"Wrong utils module imported! Expected local utils.py with 'unzip' function. "
                     f"Got: {utils_file}")

from BenchmarkProblems.SATProblem import SATProblem, SATExplainer
from config import paths

resources_dir = Path(paths.resources_dir)
problem_file_name = resources_dir / "problem_definitions" / "SAT" / "uf20-01.cnf"

problem = SATProblem.from_cnf_file(str(problem_file_name))
explainer = SATExplainer(problem)


In [3]:
# Problem and explainer are already loaded in cell 0
# This cell is kept for compatibility or can be removed
pass


In [5]:
from utils import announce
from VarianceDecisionTree.PSDecisionTree import PSDecisionTree
from Explanation.PRefManager import PRefManager

algorithm_suite = [
    "BBO"
]
print(f"Using algorithms: {', '.join(algorithm_suite)}")

with announce("generating the pRef"):
    pRef = PRefManager.generate_pRef(problem=problem,
                                     sample_size=10000,
                                     which_algorithm=" ".join(algorithm_suite))


metrics = "variance ground_truth_atomicity"
decision_tree = PSDecisionTree(maximum_depth=3,
                               ps_budget=3000,
                               ps_search_population_size=100,
                               problem=problem,
                               metrics_to_use=metrics)
with announce("training the decision tree"):
    decision_tree.train_from_pRef(pRef, verbose=True)
    

decision_tree.set_repr_ps(problem.repr_ps)

print(decision_tree.repr_long())

Using algorithms: BBO
generating the pRef...Running MEALPY BBO on SATProblem
BBO parameters: pop_size=50
   Available history attributes: ['_History__set_keyword_arguments', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'epoch', 'get_global_repeated_times', 'list_current_best', 'list_current_best_fit', 'list_current_worst', 'list_diversity', 'list_epoch_time', 'list_exploitation', 'list_exploration', 'list_global_best', 'list_global_best_fit', 'list_global_worst', 'list_population', 'log_file', 'log_to', 'logger', 'save_diversity_chart', 'save_exploration_exploitation_chart', 'save_global_best_fitness_chart', 'save_global_objectives_chart', 'save_local_best_fitness_chart', 'sav

In [7]:
import utils

for index, letter, count in zip(range(100), utils.alphabet, explainer.univariate_counts):
    print(index, letter, count)

0 A 13
1 B 11
2 C 9
3 D 13
4 E 18
5 F 8
6 G 14
7 H 9
8 I 16
9 J 15
10 K 14
11 L 17
12 M 13
13 N 14
14 O 19
15 P 11
16 Q 17
17 R 13
18 S 16
19 T 13


In [9]:
explainer.bivariate_counts

array([[13,  0,  0,  0,  3,  2,  1,  0,  3,  1,  1,  0,  1,  2,  1,  2,
         2,  1,  3,  3],
       [ 0, 11,  3,  0,  0,  1,  1,  1,  1,  1,  1,  3,  2,  0,  2,  0,
         2,  2,  1,  1],
       [ 0,  3,  9,  1,  1,  1,  0,  0,  0,  0,  3,  2,  1,  0,  2,  1,
         1,  2,  0,  0],
       [ 0,  0,  1, 13,  1,  0,  1,  2,  3,  2,  0,  1,  2,  1,  3,  2,
         0,  4,  1,  2],
       [ 3,  0,  1,  1, 18,  1,  0,  3,  3,  0,  1,  3,  3,  1,  3,  1,
         4,  2,  5,  1],
       [ 2,  1,  1,  0,  1,  8,  1,  1,  1,  1,  1,  1,  0,  1,  2,  0,
         2,  0,  0,  0],
       [ 1,  1,  0,  1,  0,  1, 14,  1,  1,  4,  1,  5,  1,  3,  1,  2,
         3,  0,  0,  2],
       [ 0,  1,  0,  2,  3,  1,  1,  9,  1,  2,  1,  2,  1,  0,  1,  0,
         1,  0,  1,  0],
       [ 3,  1,  0,  3,  3,  1,  1,  1, 16,  0,  1,  1,  3,  1,  4,  1,
         4,  0,  2,  2],
       [ 1,  1,  0,  2,  0,  1,  4,  2,  0, 15,  2,  1,  4,  2,  3,  2,
         0,  2,  3,  0],
       [ 1,  1,  3,  0,  1,  1